# Labor Market Dataset — Overview
**Project:** Historical European & US Labor Market Analysis (2010–2024)  
**Scope:** IT · Finance · HR sectors  
**Sources:** Wayback Machine CDX API + direct scraping  
**Goal:** ~500k clean job postings for NLP-based skills demand analysis

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re
import warnings
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# ── Style ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'sans-serif',
})
PALETTE = ['#2563eb', '#16a34a', '#dc2626', '#d97706', '#7c3aed', '#0891b2', '#be185d']

DATA_PATH = '../../data/processed/*.csv'
print('Setup complete.')

---
## 1. Dataset Inventory

In [ ]:
files = glob.glob(DATA_PATH)
print(f'Found {len(files)} CSV files:\n')
for f in sorted(files):
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {os.path.basename(f):<45}  {size_mb:6.1f} MB')

---
## 2. Source Summary Table

In [ ]:
# Metadata for each source
SOURCE_META = {
    'stackoverflow':      {'country': 'Global',      'sector': 'IT',      'region': 'Global'},
    'efinancialcareers':  {'country': 'Global',      'sector': 'Finance', 'region': 'Global'},
    'stepstone_de':       {'country': 'Germany',     'sector': 'Mixed',   'region': 'Europe'},
    'pracuj_pl':          {'country': 'Poland',      'sector': 'Mixed',   'region': 'Europe'},
    'ch_data':            {'country': 'Switzerland', 'sector': 'Mixed',   'region': 'Europe'},
    'dice_com':           {'country': 'USA',         'sector': 'IT',      'region': 'North America'},
    'monster_com':        {'country': 'USA',         'sector': 'Mixed',   'region': 'North America'},
}

summary_stats = []

for file in sorted(files):
    filename = os.path.basename(file)
    try:
        df = pd.read_csv(file, low_memory=False)
        total_rows = len(df)

        # Description column
        desc_col = next(
            (c for c in df.columns if c.lower() in
             ['description', 'description_preview', 'job_description', 'text', 'full_text']),
            None
        )
        if desc_col:
            valid_desc = df[desc_col].notna() & (df[desc_col].astype(str).str.strip() != '')
            n_valid = valid_desc.sum()
        else:
            n_valid = 0

        # Date column
        date_col = next(
            (c for c in df.columns if 'date' in c.lower() or c.lower() == 'timestamp'),
            None
        )
        if date_col:
            dates = pd.to_datetime(df[date_col], errors='coerce', utc=True).dropna()
            min_date = dates.min().strftime('%Y-%m-%d') if not dates.empty else 'N/A'
            max_date = dates.max().strftime('%Y-%m-%d') if not dates.empty else 'N/A'
            years_covered = len(dates.dt.year.unique()) if not dates.empty else 0
        else:
            min_date, max_date, years_covered = 'N/A', 'N/A', 0

        # Avg description length
        if desc_col and n_valid > 0:
            avg_len = int(df.loc[valid_desc, desc_col].astype(str).str.len().mean())
        else:
            avg_len = 0

        slug = (filename
                .replace('_jobs_dedup.csv', '')
                .replace('_dedup.csv', '')
                .replace('_cleaned.csv', '')
                .replace('_jobs.csv', '')
                .replace('.csv', ''))

        meta = SOURCE_META.get(slug, {'country': '?', 'sector': '?', 'region': '?'})
        missing_pct = round((total_rows - n_valid) / total_rows * 100, 1) if total_rows > 0 else 0

        summary_stats.append({
            'Source':           slug,
            'Country':          meta['country'],
            'Sector':           meta['sector'],
            'Total Vacancies':  total_rows,
            'Valid Desc':       n_valid,
            'Missing Desc (%)': missing_pct,
            'Avg Desc Length':  avg_len,
            'Oldest Record':    min_date,
            'Newest Record':    max_date,
            'Years Covered':    years_covered,
        })

    except Exception as e:
        print(f'Error: {filename} — {e}')

overview_df = pd.DataFrame(summary_stats)
overview_df = overview_df.sort_values('Total Vacancies', ascending=False).reset_index(drop=True)

total_vac  = overview_df['Total Vacancies'].sum()
total_desc = overview_df['Valid Desc'].sum()

print(f'Total vacancies:            {total_vac:>10,}')
print(f'With valid descriptions:    {total_desc:>10,}  ({total_desc/total_vac*100:.1f}%)')
print(f'Sources:                    {len(overview_df):>10}')
print()
display(overview_df)

---
## 3. Volume by Source

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: total vs valid desc ───────────────────────────
ax = axes[0]
x = np.arange(len(overview_df))
w = 0.35
ax.bar(x - w/2, overview_df['Total Vacancies'],  w, label='Total',          color='#93c5fd')
ax.bar(x + w/2, overview_df['Valid Desc'],        w, label='Valid Desc',     color='#2563eb')
ax.set_xticks(x)
ax.set_xticklabels(overview_df['Source'], rotation=35, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title('Total Vacancies vs Valid Descriptions')
ax.legend()

# ── Right: description fill rate ───────────────────────
ax2 = axes[1]
colors = ['#16a34a' if v >= 80 else '#d97706' if v >= 40 else '#dc2626'
          for v in (100 - overview_df['Missing Desc (%)'])]
bars = ax2.barh(overview_df['Source'][::-1],
                100 - overview_df['Missing Desc (%)'].values[::-1],
                color=colors[::-1])
ax2.axvline(80, color='gray', linestyle='--', linewidth=0.8, label='80% threshold')
ax2.set_xlim(0, 105)
ax2.set_xlabel('Description Fill Rate (%)')
ax2.set_title('Description Fill Rate by Source')
ax2.legend(fontsize=8)
for bar, val in zip(bars, (100 - overview_df['Missing Desc (%)'].values[::-1])):
    ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
             f'{val:.0f}%', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig1_volume_overview.png', bbox_inches='tight')
plt.show()

---
## 4. Temporal Coverage

In [ ]:
# Build yearly counts per source
yearly_data = {}

for file in sorted(files):
    filename = os.path.basename(file)
    slug = (filename
            .replace('_jobs_dedup.csv', '')
            .replace('_dedup.csv', '')
            .replace('_cleaned.csv', '')
            .replace('_jobs.csv', '')
            .replace('.csv', ''))
    try:
        df = pd.read_csv(file, low_memory=False)
        date_col = next(
            (c for c in df.columns if 'date' in c.lower() or c.lower() == 'timestamp'),
            None
        )
        if date_col:
            years = pd.to_datetime(df[date_col], errors='coerce', utc=True).dt.year.dropna()
            yearly_data[slug] = years.value_counts().sort_index()
    except:
        pass

# Combine into one DataFrame
yearly_df = pd.DataFrame(yearly_data).fillna(0).astype(int)
yearly_df = yearly_df.loc[(yearly_df.index >= 2010) & (yearly_df.index <= 2025)]

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# ── Top: stacked area ──────────────────────────────────
ax = axes[0]
yearly_df.plot.area(ax=ax, stacked=True, color=PALETTE[:len(yearly_df.columns)], alpha=0.85)
ax.set_title('Vacancies per Year by Source (stacked)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.set_xlim(yearly_df.index.min(), yearly_df.index.max())

# ── Bottom: heatmap of coverage ─────────────────────────
ax2 = axes[1]
heatmap_data = yearly_df.T
# Normalize each source to 0-1 for visibility
heatmap_norm = heatmap_data.div(heatmap_data.max(axis=1).replace(0, 1), axis=0)
sns.heatmap(
    heatmap_norm,
    ax=ax2,
    cmap='Blues',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Relative volume (per source)'},
    annot=heatmap_data.applymap(lambda v: f'{int(v/1000)}k' if v >= 1000 else str(int(v)) if v > 0 else ''),
    fmt='',
    annot_kws={'size': 7}
)
ax2.set_title('Temporal Coverage Heatmap (normalized per source)')
ax2.set_xlabel('Year')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fig2_temporal_coverage.png', bbox_inches='tight')
plt.show()

---
## 5. Geographic & Sector Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── By Country ─────────────────────────────────────────
country_agg = overview_df.groupby('Country')['Total Vacancies'].sum().sort_values(ascending=False)
axes[0].bar(country_agg.index, country_agg.values,
            color=PALETTE[:len(country_agg)])
axes[0].set_title('Vacancies by Country')
axes[0].set_xticklabels(country_agg.index, rotation=30, ha='right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))

# ── By Sector ──────────────────────────────────────────
sector_agg = overview_df.groupby('Sector')['Total Vacancies'].sum().sort_values(ascending=False)
axes[1].pie(
    sector_agg.values,
    labels=sector_agg.index,
    autopct='%1.1f%%',
    colors=PALETTE[:len(sector_agg)],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Vacancies by Sector')

# ── Description quality by source ─────────────────────
fill_rate = 100 - overview_df.set_index('Source')['Missing Desc (%)']
avg_len   = overview_df.set_index('Source')['Avg Desc Length']
ax3 = axes[2]
sc = ax3.scatter(
    fill_rate.values,
    avg_len.values,
    s=overview_df['Total Vacancies'].values / 1000,
    c=PALETTE[:len(fill_rate)],
    alpha=0.8,
    edgecolors='white',
    linewidths=0.5
)
for i, (src, x, y) in enumerate(zip(fill_rate.index, fill_rate.values, avg_len.values)):
    ax3.annotate(src, (x, y), fontsize=7.5,
                 xytext=(4, 4), textcoords='offset points')
ax3.axvline(80, color='gray', linestyle='--', linewidth=0.8)
ax3.set_xlabel('Description Fill Rate (%)')
ax3.set_ylabel('Avg Description Length (chars)')
ax3.set_title('Description Quality\n(bubble = dataset size)')

plt.tight_layout()
plt.savefig('fig3_geo_sector.png', bbox_inches='tight')
plt.show()

---
## 6. Per-Source Deep Dive

In [ ]:
for file in sorted(files):
    filename = os.path.basename(file)
    slug = (filename
            .replace('_jobs_dedup.csv', '')
            .replace('_dedup.csv', '')
            .replace('_cleaned.csv', '')
            .replace('_jobs.csv', '')
            .replace('.csv', ''))
    try:
        df = pd.read_csv(file, low_memory=False)

        desc_col = next(
            (c for c in df.columns if c.lower() in
             ['description', 'description_preview', 'job_description', 'text', 'full_text']),
            None
        )
        date_col = next(
            (c for c in df.columns if 'date' in c.lower() or c.lower() == 'timestamp'),
            None
        )

        print(f'\n{"="*55}')
        print(f'  {slug.upper()}')
        print(f'{"="*55}')
        print(f'  Rows:     {len(df):,}')
        print(f'  Columns:  {list(df.columns)}')

        # Null rates for all columns
        null_pct = (df.isnull().sum() / len(df) * 100).round(1)
        print(f'\n  Null rates (%):')
        for col, pct in null_pct.items():
            bar = '█' * int((100 - pct) / 5) + '░' * int(pct / 5)
            print(f'    {col:<25} {bar}  {100-pct:.0f}% filled')

        # Yearly distribution
        if date_col:
            years = pd.to_datetime(df[date_col], errors='coerce', utc=True).dt.year.dropna()
            if not years.empty:
                print(f'\n  Records per year:')
                vc = years.value_counts().sort_index()
                for yr, cnt in vc.items():
                    bar = '█' * (cnt * 30 // vc.max())
                    print(f'    {int(yr)}  {bar:<31} {cnt:,}')

        # Sample descriptions
        if desc_col:
            valid = df[desc_col].dropna()
            valid = valid[valid.astype(str).str.strip() != '']
            if not valid.empty:
                sample = valid.sample(min(2, len(valid)), random_state=42).iloc[0]
                print(f'\n  Sample description (first 300 chars):')
                print(f'  "{str(sample)[:300]}..."')

    except Exception as e:
        print(f'Error: {slug} — {e}')

---
## 7. Description Length Distribution

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, file in enumerate(sorted(files)):
    if i >= len(axes):
        break
    filename = os.path.basename(file)
    slug = (filename
            .replace('_jobs_dedup.csv', '')
            .replace('_dedup.csv', '')
            .replace('_cleaned.csv', '')
            .replace('_jobs.csv', '')
            .replace('.csv', ''))
    try:
        df = pd.read_csv(file, low_memory=False)
        desc_col = next(
            (c for c in df.columns if c.lower() in
             ['description', 'description_preview', 'job_description', 'text', 'full_text']),
            None
        )
        if desc_col:
            lengths = df[desc_col].dropna().astype(str).str.len()
            lengths = lengths[lengths > 50].clip(upper=10000)
            if not lengths.empty:
                axes[i].hist(lengths, bins=50, color=PALETTE[i % len(PALETTE)], alpha=0.8, edgecolor='white')
                axes[i].axvline(lengths.median(), color='black', linestyle='--',
                                linewidth=1, label=f'median={int(lengths.median()):,}')
                axes[i].set_title(slug, fontsize=9)
                axes[i].set_xlabel('chars', fontsize=8)
                axes[i].legend(fontsize=7)
                axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))
        else:
            axes[i].text(0.5, 0.5, 'no desc column', ha='center', va='center',
                        transform=axes[i].transAxes, color='gray')
            axes[i].set_title(slug, fontsize=9)
    except Exception as e:
        axes[i].set_title(f'{slug}\nERROR', fontsize=9, color='red')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Description Length Distribution by Source', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig4_desc_lengths.png', bbox_inches='tight')
plt.show()

---
## 8. Data Quality Flags

In [ ]:
quality_issues = []

for _, row in overview_df.iterrows():
    issues = []
    fill = 100 - row['Missing Desc (%)']

    if fill < 40:
        issues.append(f'⚠ Low description fill rate ({fill:.0f}%)')
    elif fill < 80:
        issues.append(f'△ Moderate description fill rate ({fill:.0f}%)')

    if row['Avg Desc Length'] > 0 and row['Avg Desc Length'] < 300:
        issues.append(f'⚠ Short avg description ({row["Avg Desc Length"]} chars)')

    if row['Years Covered'] <= 2:
        issues.append(f'△ Narrow time window ({row["Years Covered"]} year(s))')

    if row['Total Vacancies'] < 5000:
        issues.append(f'△ Small dataset ({row["Total Vacancies"]:,} records)')

    quality_issues.append({
        'Source':  row['Source'],
        'Country': row['Country'],
        'Fill %':  f"{fill:.0f}%",
        'Status':  '✅ Good' if not issues else ' | '.join(issues)
    })

quality_df = pd.DataFrame(quality_issues)
print('Data Quality Flags:\n')
display(quality_df)

---
## 9. Usable Dataset for ML Pipeline

In [ ]:
# Define what counts as "usable" for the ML pipeline
# Criteria: fill rate >= 50% AND total vacancies >= 5000
FILL_THRESHOLD = 50
SIZE_THRESHOLD = 5000

overview_df['Fill Rate (%)'] = 100 - overview_df['Missing Desc (%)']
usable = overview_df[
    (overview_df['Fill Rate (%)'] >= FILL_THRESHOLD) &
    (overview_df['Total Vacancies'] >= SIZE_THRESHOLD)
].copy()

excluded = overview_df[
    ~overview_df['Source'].isin(usable['Source'])
].copy()

usable_total  = usable['Valid Desc'].sum()
usable_vac    = usable['Total Vacancies'].sum()

print('Sources included in ML pipeline:')
print(f'  Criteria: fill rate ≥ {FILL_THRESHOLD}% AND size ≥ {SIZE_THRESHOLD:,}')
print()
display(usable[['Source', 'Country', 'Sector', 'Total Vacancies',
                'Valid Desc', 'Fill Rate (%)', 'Oldest Record', 'Newest Record']])

print(f'\nUsable vacancies (total):         {usable_vac:,}')
print(f'Usable with valid descriptions:   {usable_total:,}')

if not excluded.empty:
    print(f'\nExcluded sources:')
    for _, row in excluded.iterrows():
        fill = 100 - row['Missing Desc (%)']
        reason = []
        if fill < FILL_THRESHOLD:    reason.append(f'fill={fill:.0f}%')
        if row['Total Vacancies'] < SIZE_THRESHOLD: reason.append(f'size={row["Total Vacancies"]:,}')
        print(f'  {row["Source"]:<25} ({" | ".join(reason)})')

---
## 10. Summary for Project Proposal

In [ ]:
print('=' * 58)
print('  DATASET SUMMARY — LABOR MARKET ANALYSIS PROJECT')
print('=' * 58)
print()
print(f'  Total sources scraped:          {len(overview_df)}')
print(f'  Total raw vacancies:            {total_vac:,}')
print(f'  Vacancies with descriptions:    {total_desc:,}')
print(f'  Sources usable for ML:          {len(usable)}')
print(f'  Usable records (valid desc):    {usable_total:,}')
print()
print('  Geographic coverage:')
for country in overview_df['Country'].unique():
    n = overview_df[overview_df['Country'] == country]['Total Vacancies'].sum()
    print(f'    {country:<18} {n:>8,}')
print()
print('  Time span:  2010 – 2024')
print('  Languages:  English, German, Polish, French')
print()
print('  ML Pipeline plan:')
print('    Phase 1 — Ground truth labeling  (IT / Finance / HR / Other)')
print('    Phase 2 — Domain classification  (DistilBERT-multilingual)')
print('    Phase 3 — Skills extraction      (PhraseMatcher + ESCO taxonomy)')
print('    Phase 4 — Time-series analysis   (skills lifecycle 2010–2024)')
print()
print('=' * 58)